# Préparation des données

### Importation des librairies

In [ ]:
!pip install scikit-learn==1.6.1

In [1]:
# Librairies de base
import pandas as pd
import numpy as np
import re
import nltk
import joblib
# NLP
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score

from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

# Pour sauvegarder le modèle
import pickle


### Prétraitement des données

In [2]:

# Chemin du fichier Excel
file_path = 'Classeur_des_donnees_sante.xlsx'

# Obtenir les noms des feuilles
xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names

# Voir les feuilles
print(sheet_names)

['Grossesse', 'Accouchement', 'Visite_Medicale_Vaccination', 'Conseils']


In [3]:

# Créer une liste pour stocker les DataFrames
data = []

# Boucle à travers chaque feuille
for sheet in sheet_names:
    # Lire la feuille dans un DataFrame
    df = pd.read_excel(xls, sheet_name=sheet)
    
    # Vérifier si les colonnes nécessaires existent
    #if 'Question' in df.columns and 'Reponse' in df.columns:
    if 'Theme' in df.columns and 'Question' in df.columns and 'Reponse' in df.columns:
        # Créer un DataFrame avec les colonnes nécessaires
        df_filtered = pd.DataFrame({
            #'Theme': sheet,
            'Theme': df['Theme'],
            'Question': df['Question'],
            'Reponse': df['Reponse']
        })
        data.append(df_filtered)

# Combiner tous les DataFrames
result = pd.concat(data, ignore_index=True)
# Transformer les valeurs de la colonne 'thème'
#result['Theme'] = result['Theme'].replace('Visite_Medicale_Vaccination', 'Vaccination')

# Sauvegarder le résultat dans un fichier CSV
result.to_csv('donnees_regroupees.csv', index=False, encoding='utf-8')

print("Données regroupées et sauvegardées dans 'donnees_regroupees.csv' avec succès !")

Données regroupées et sauvegardées dans 'donnees_regroupees.csv' avec succès !


### Charger le dataset 

In [4]:
df = pd.read_csv("donnees_regroupees.csv")

In [5]:
df.describe()

,Theme,Question,Reponse
count,130,130,130
unique,4,129,130
top,Grossesse,Quand mon bébé doit-il voir un agent de santé ...,Vous devez plus vous reposer et manger davant...
freq,51,2,1


# Préparation du pipeline et du Model

## Pipeline

### Prétraiter les données

In [6]:
# Prétraitement avec SpaCy
nlp = spacy.load("fr_core_news_sm")
def preprocess(text):
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return " ".join(tokens)

df["question_clean"] = df["Question"].apply(preprocess)


### Séparer train/test pour le modèle ML

In [7]:

X = df["question_clean"]
y = df["Theme"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


### Entraîner Naive Bayes sur le thème

In [8]:
model_theme = make_pipeline(TfidfVectorizer(), MultinomialNB())
model_theme.fit(X_train, y_train)


Pipeline(steps=[('tfidfvectorizer', TfidfVectorizer()),
                ('multinomialnb', MultinomialNB())])

### Évaluer la précision du modèle sur le thème

In [9]:

y_pred = model_theme.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Précision sur le thème : {accuracy*100:.2f}%")

Précision sur le thème : 76.92%


### Préparer le vecteur TF-IDF pour recherche exacte

In [10]:
vectorizer_all = TfidfVectorizer()
X_vect_all = vectorizer_all.fit_transform(df["question_clean"])


### Une fonction de mapping assocciée

In [11]:
def get_response(user_question):
    # Prétraitement
    user_question_clean = preprocess(user_question)
    
    #  Prédire le thème
    predicted_theme = model_theme.predict([user_question_clean])[0]
    
    #  Filtrer les questions du même thème
    df_theme = df[df["Theme"] == predicted_theme].reset_index(drop=True)
    
    #  Calculer TF-IDF pour ce sous-ensemble
    vectorizer_theme = TfidfVectorizer()
    X_vect_theme = vectorizer_theme.fit_transform(df_theme["question_clean"])
    user_vect = vectorizer_theme.transform([user_question_clean])
    
    #  Similarité cosinus
    sim_scores = cosine_similarity(user_vect, X_vect_theme)
    best_idx = sim_scores.argmax()
    
    #  Retourner la réponse
    return df_theme.iloc[best_idx]["Reponse"]


### Tester le chatbot

In [12]:
questions_test = [
    "Quels sont les signes précoces de grossesse ?",
    "Quand vacciner un nourrisson ?",
    "Comment nourrir mon bébé correctement ?",
    "Comment faire un positionnement correct du bébé pendant l'allaitement?"
]

for q in questions_test:
    print("Question :", q)
    print("Réponse :", get_response(q))
    print("---")


Question : Quels sont les signes précoces de grossesse ?
Réponse : Les signes de danger incluent des saignements abondants, des douleurs abdominales sévères, des maux de tête persistants, une augmentation soudaine du gonflement des doigts, du visage et des jambes, et tout changement de mouvement du fœtus.
---
Question : Quand vacciner un nourrisson ?
Réponse : Lavez-vous les mains, évitez la foule et respectez le calendrier vaccinal.
---
Question : Comment nourrir mon bébé correctement ?
Réponse : Le nouveau-né doit être nourri exclusivement au sein pendant les 6 premiers mois si possible.
---
Question : Comment faire un positionnement correct du bébé pendant l'allaitement?
Réponse :  Le positionnement est correct quand : • la tête et le corps de l’enfant sont bien soutenus et près du corps de la mère • son visage et son ventre sont face à la mère • son oreille et son épaule sont dans le même axe, le cou n’étant pas tordu.  • la bouche de l’enfant recouvre la plupart de l’aréole (la pa

## Sauvegarder le modèle et les vectorizers pour Flask

In [13]:
joblib.dump(model_theme, "chatbot/model_theme.pkl")
joblib.dump(vectorizer_all, "chatbot/vectorizer_all.pkl")
joblib.dump(df, "chatbot/intents_df.pkl")

['chatbot/intents_df.pkl']